# 03 — Concurrent.futures

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- utiliser `ThreadPoolExecutor` et `ProcessPoolExecutor` via une API unifiée ;
- soumettre des tâches avec `submit()` et récupérer les résultats via `Future` ;
- utiliser `as_completed()` et `map()` pour orchestrer les résultats ;
- gérer les exceptions dans les `Future` ;
- choisir entre threads et processus selon le type de tâche.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :

- `threading.Thread`, `Lock`, `Event` et le GIL ;
- `multiprocessing.Process`, `Pool`, `Queue` et `shared_memory` ;
- les gestionnaires de contexte (`with`) ;
- les itérateurs et la programmation fonctionnelle.

Notions que nous allons **introduire** ici :

- `concurrent.futures` : `ThreadPoolExecutor`, `ProcessPoolExecutor` ;
- l'objet `Future` et ses méthodes ;
- `as_completed()`, `wait()`, et `Executor.map()`.

## Plan

1. Pourquoi `concurrent.futures` ?
2. `ThreadPoolExecutor` — pool de threads
3. `ProcessPoolExecutor` — pool de processus
4. L'objet `Future`
5. `as_completed()` — résultats dès qu'ils arrivent
6. `wait()` — attente conditionnelle
7. `Executor.map()` — map parallèle
8. Gestion des exceptions
9. Patterns avancés
10. Synthèse
11. Exercices

---

## 1. Pourquoi `concurrent.futures` ?

`concurrent.futures` (PEP 3148) fournit une **API unifiée** pour le parallélisme en threads et en processus. L'idée : vous écrivez votre code une fois, et vous choisissez l'exécuteur (threads ou processus) selon le type de tâche.

| Avantage | Détail |
|---|---|
| API unique | Même code pour threads et processus |
| `Future` | Représente un résultat futur, inspectable |
| `as_completed` | Itère sur les résultats dès qu'ils arrivent |
| Gestionnaire de contexte | `with` s'occupe du shutdown |

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print(f"ThreadPoolExecutor : {ThreadPoolExecutor}")
print(f"ProcessPoolExecutor : {ProcessPoolExecutor}")

---

## 2. `ThreadPoolExecutor` — pool de threads

Idéal pour les tâches **I/O-bound** (requêtes réseau, lecture de fichiers, etc.).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def telecharger(url: str) -> str:
    time.sleep(0.5)  # simule I/O
    return f"{url} : OK"

urls = [f"https://example.com/page{i}" for i in range(8)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    resultats = list(executor.map(telecharger, urls))
elapsed = time.perf_counter() - start

for r in resultats:
    print(r)
print(f"Temps : {elapsed:.2f}s (au lieu de {0.5 * len(urls):.1f}s séquentiel)")

### Nombre de workers par défaut

Depuis Python 3.8, `ThreadPoolExecutor` utilise `min(32, os.cpu_count() + 4)` workers par défaut.

In [ ]:
import os

default_workers = min(32, (os.cpu_count() or 1) + 4)
print(f"Workers par défaut : {default_workers}")

---

## 3. `ProcessPoolExecutor` — pool de processus

Idéal pour les tâches **CPU-bound**. L'API est identique.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def calcul_lourd(n: int) -> int:
    return sum(i * i for i in range(n))

donnees = [5_000_000] * 8

# Séquentiel
start = time.perf_counter()
seq = [calcul_lourd(n) for n in donnees]
t_seq = time.perf_counter() - start

# Parallèle
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    par = list(executor.map(calcul_lourd, donnees))
t_par = time.perf_counter() - start

print(f"Séquentiel : {t_seq:.3f}s")
print(f"Pool(4)    : {t_par:.3f}s")
print(f"Accélération : {t_seq / t_par:.1f}x")

### Changer d'exécuteur en une ligne

C'est la force de l'API unifiée : il suffit de changer la classe.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def traiter(x: int) -> int:
    return x * x

# Bascule en changeant simplement la classe
ExecutorClass = ThreadPoolExecutor  # ou ProcessPoolExecutor

with ExecutorClass(max_workers=4) as executor:
    resultats = list(executor.map(traiter, range(10)))
print(resultats)

---

## 4. L'objet `Future`

Un `Future` représente un **résultat futur**. On l'obtient avec `executor.submit()`. Contrairement à `map()`, `submit()` retourne un `Future` par tâche, ce qui permet un contrôle plus fin.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def calcul(x: int) -> int:
    time.sleep(0.5)
    return x * x

with ThreadPoolExecutor(max_workers=2) as executor:
    future = executor.submit(calcul, 42)
    print(f"Type : {type(future)}")
    print(f"En cours : {future.running()}")
    print(f"Terminé : {future.done()}")
    resultat = future.result()  # bloque jusqu'au résultat
    print(f"Résultat : {resultat}")
    print(f"Terminé : {future.done()}")

### Méthodes de `Future`

| Méthode | Description |
|---|---|
| `result(timeout=None)` | Attend et retourne le résultat |
| `exception(timeout=None)` | Attend et retourne l'exception (ou `None`) |
| `done()` | `True` si la tâche est terminée |
| `running()` | `True` si en cours d'exécution |
| `cancelled()` | `True` si annulée |
| `cancel()` | Tente d'annuler (pas toujours possible) |
| `add_done_callback(fn)` | Callback appelé à la fin |

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def longue_tache(x: int) -> int:
    time.sleep(1)
    return x * 10

def callback(future):
    print(f"Callback : résultat = {future.result()}")

with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(longue_tache, 7)
    future.add_done_callback(callback)
    print("Callback enregistré, j'attends...")

---

## 5. `as_completed()` — résultats dès qu'ils arrivent

`as_completed()` itère sur les `Future` dans l'ordre où ils **terminent**, pas dans l'ordre de soumission. C'est le pattern le plus utile de `concurrent.futures`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random

def tache(n: int) -> tuple[int, float]:
    duree = random.uniform(0.1, 1.0)
    time.sleep(duree)
    return n, duree

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(tache, i): i for i in range(8)}

    for future in as_completed(futures):
        n, duree = future.result()
        print(f"Tâche {n} terminée en {duree:.2f}s")

### Pattern : associer un `Future` à ses métadonnées

On utilise un dictionnaire `{future: metadata}` pour retrouver quelle tâche correspond à quel `Future`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def telecharger(url: str) -> str:
    time.sleep(0.3)
    return f"Contenu de {url}"

urls = [f"https://api.example.com/item/{i}" for i in range(6)]

with ThreadPoolExecutor(max_workers=3) as executor:
    future_to_url = {executor.submit(telecharger, url): url for url in urls}

    for future in as_completed(future_to_url):
        url = future_to_url[future]
        try:
            data = future.result()
            print(f"{url} → {data}")
        except Exception as e:
            print(f"{url} → ERREUR : {e}")

---

## 6. `wait()` — attente conditionnelle

`wait()` attend un ensemble de `Future` avec un mode configurable : `FIRST_COMPLETED`, `FIRST_EXCEPTION`, ou `ALL_COMPLETED`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import time
import random

def tache(n: int) -> int:
    time.sleep(random.uniform(0.1, 0.5))
    return n

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(tache, i) for i in range(6)]

    done, not_done = wait(futures, return_when=FIRST_COMPLETED)
    print(f"Premières terminées : {len(done)}")
    print(f"En attente : {len(not_done)}")
    for f in done:
        print(f"  Résultat : {f.result()}")

---

## 7. `Executor.map()` — map parallèle

`map()` est plus simple que `submit()` + `as_completed()` quand on veut les résultats dans l'**ordre de soumission**.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def doubler(x: int) -> int:
    time.sleep(0.1)
    return x * 2

with ThreadPoolExecutor(max_workers=4) as executor:
    resultats = executor.map(doubler, range(10))
    # map retourne un itérateur, pas une liste
    print(type(resultats))
    print(list(resultats))

### `map()` avec `chunksize`

Pour `ProcessPoolExecutor`, `chunksize` réduit l'overhead de communication en envoyant des lots au lieu d'éléments individuels.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def carre(x: int) -> int:
    return x * x

data = list(range(100_000))

start = time.perf_counter()
with ProcessPoolExecutor(4) as executor:
    r1 = list(executor.map(carre, data, chunksize=1))
t1 = time.perf_counter() - start

start = time.perf_counter()
with ProcessPoolExecutor(4) as executor:
    r2 = list(executor.map(carre, data, chunksize=1000))
t2 = time.perf_counter() - start

print(f"chunksize=1    : {t1:.3f}s")
print(f"chunksize=1000 : {t2:.3f}s")
print(f"Accélération   : {t1/t2:.1f}x")

---

## 8. Gestion des exceptions

Les exceptions dans un worker sont capturées par le `Future`. Elles sont relancées quand on appelle `result()`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def risque(x: int) -> float:
    if x == 0:
        raise ValueError("Division par zéro")
    return 1.0 / x

with ThreadPoolExecutor(max_workers=2) as executor:
    future = executor.submit(risque, 0)

    # L'exception est stockée dans le Future
    print(f"Exception : {future.exception()}")

    # Elle est relancée avec result()
    try:
        future.result()
    except ValueError as e:
        print(f"Attrapée : {e}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def tache_risquee(x: int) -> str:
    if x % 3 == 0:
        raise RuntimeError(f"Erreur pour x={x}")
    return f"OK-{x}"

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(tache_risquee, i): i for i in range(10)}
    for future in as_completed(futures):
        idx = futures[future]
        try:
            print(f"  {idx} → {future.result()}")
        except RuntimeError as e:
            print(f"  {idx} → ERREUR : {e}")

---

## 9. Patterns avancés

### 9.1. Annulation de tâches

On peut annuler un `Future` **avant** qu'il ne commence. Une tâche en cours ne peut pas être annulée.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def lente(x: int) -> int:
    time.sleep(2)
    return x

with ThreadPoolExecutor(max_workers=1) as executor:
    f1 = executor.submit(lente, 1)  # occupe le worker
    f2 = executor.submit(lente, 2)  # en file d'attente

    annule = f2.cancel()
    print(f"f2 annulé : {annule}")
    print(f"f2 cancelled : {f2.cancelled()}")

### 9.2. Rate limiter avec `Semaphore`

Pour limiter le nombre de requêtes simultanées à une API externe.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time

semaphore = threading.Semaphore(2)  # max 2 requêtes simultanées

def requete_limitee(url: str) -> str:
    with semaphore:
        print(f"Début {url}")
        time.sleep(0.5)
        return f"{url} → OK"

urls = [f"api/{i}" for i in range(6)]

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(requete_limitee, url) for url in urls]
    for f in as_completed(futures):
        print(f"  {f.result()}")

### 9.3. Timeout global

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def lent() -> str:
    time.sleep(5)
    return "fini"

with ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(lent)
    try:
        resultat = future.result(timeout=1)
    except TimeoutError:
        print("Timeout ! La tâche est trop lente.")

---

## 10. Synthèse

| Outil | Usage |
|---|---|
| `ThreadPoolExecutor` | I/O-bound (réseau, fichiers) |
| `ProcessPoolExecutor` | CPU-bound (calcul) |
| `submit(fn, *args)` | Retourne un `Future` |
| `map(fn, iterable)` | Résultats dans l'ordre |
| `as_completed(futures)` | Résultats dès qu'ils arrivent |
| `wait(futures)` | Attente conditionnelle |
| `future.result()` | Attend et retourne (ou relance l'exception) |
| `future.cancel()` | Annule si pas encore démarré |
| `future.add_done_callback()` | Callback asynchrone |

**Quand utiliser quoi ?**

- Résultats dans l'ordre → `map()`
- Résultats au plus tôt → `submit()` + `as_completed()`
- Besoin de contrôle fin → `submit()` + `Future`

---

## 11. Exercices

### Exercice 1 — Scraping parallèle simulé *(facile)*

Simuler le téléchargement de 10 pages web (chaque téléchargement = `time.sleep(random.uniform(0.1, 0.5))`) avec un `ThreadPoolExecutor(4)` et `as_completed()`. Afficher chaque résultat dès qu'il arrive.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Concurrent_futures", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random

def telecharger(url: str) -> str:
    duree = random.uniform(0.1, 0.5)
    time.sleep(duree)
    return f"{url} ({duree:.2f}s)"

urls = [f"page-{i}.html" for i in range(10)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(telecharger, url): url for url in urls}
    for future in as_completed(futures):
        print(f"  {future.result()}")
print(f"Total : {time.perf_counter() - start:.2f}s")
```

</details>

### Exercice 2 — Calcul distribué de nombres premiers *(moyen)*

Écrire une fonction `est_premier(n: int) -> bool`. Utiliser `ProcessPoolExecutor` et `map()` pour tester tous les nombres de 2 à 100 000. Collecter les nombres premiers. Comparer avec le temps séquentiel.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Concurrent_futures", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from concurrent.futures import ProcessPoolExecutor
import time
import math

def est_premier(n: int) -> bool:
    if n < 2:
        return False
    if n < 4:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(math.isqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

N = 100_000
nombres = range(2, N + 1)

# Séquentiel
start = time.perf_counter()
premiers_seq = [n for n in nombres if est_premier(n)]
t_seq = time.perf_counter() - start

# Parallèle
start = time.perf_counter()
with ProcessPoolExecutor(4) as executor:
    flags = list(executor.map(est_premier, nombres, chunksize=1000))
premiers_par = [n for n, ok in zip(nombres, flags) if ok]
t_par = time.perf_counter() - start

print(f"Premiers trouvés : {len(premiers_par)}")
print(f"Séquentiel : {t_seq:.3f}s")
print(f"Parallèle  : {t_par:.3f}s")
assert premiers_seq == premiers_par
```

</details>

### Exercice 3 — Retry avec timeout *(difficile)*

Écrire une fonction `execute_with_retry(executor, fn, args, max_retries=3, timeout=2.0)` qui :

1. Soumet la tâche avec `submit()`.
2. Attend le résultat avec un timeout.
3. En cas de `TimeoutError` ou d'exception, réessaie (jusqu'à `max_retries`).
4. Retourne le résultat ou lève la dernière exception.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Concurrent_futures", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from concurrent.futures import ThreadPoolExecutor, Future
import time
import random

def execute_with_retry(
    executor: ThreadPoolExecutor,
    fn,
    args: tuple = (),
    max_retries: int = 3,
    timeout: float = 2.0,
):
    last_exc = None
    for attempt in range(1, max_retries + 1):
        future = executor.submit(fn, *args)
        try:
            return future.result(timeout=timeout)
        except (TimeoutError, Exception) as e:
            last_exc = e
            print(f"  Tentative {attempt}/{max_retries} échouée : {e}")
    raise last_exc

# Test : une fonction qui échoue aléatoirement
def flaky(x: int) -> str:
    if random.random() < 0.5:
        raise RuntimeError("Erreur aléatoire")
    return f"OK-{x}"

with ThreadPoolExecutor(max_workers=2) as executor:
    try:
        resultat = execute_with_retry(executor, flaky, (42,), max_retries=5)
        print(f"Résultat : {resultat}")
    except RuntimeError as e:
        print(f"Échec final : {e}")
```

</details>

### Exercice 4 — Pipeline parallèle *(difficile)*

Implémenter un pipeline en 3 étapes :

1. **Charger** : simule la lecture d'un fichier (`ThreadPoolExecutor`).
2. **Transformer** : calcul CPU (`ProcessPoolExecutor`).
3. **Sauvegarder** : simule l'écriture (`ThreadPoolExecutor`).

Utiliser `as_completed()` pour chaîner les étapes : dès qu'un chargement finit, soumettre la transformation, etc.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Concurrent_futures", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import time

def charger(nom: str) -> tuple[str, list[int]]:
    time.sleep(0.2)  # I/O
    return nom, list(range(1000))

def transformer(data: tuple[str, list[int]]) -> tuple[str, int]:
    nom, valeurs = data
    return nom, sum(v * v for v in valeurs)  # CPU

def sauvegarder(result: tuple[str, int]) -> str:
    nom, total = result
    time.sleep(0.1)  # I/O
    return f"{nom}: {total}"

fichiers = [f"data_{i}.csv" for i in range(8)]

start = time.perf_counter()
with ThreadPoolExecutor(4) as io_pool, ProcessPoolExecutor(4) as cpu_pool:
    # Étape 1 : charger
    load_futures = {io_pool.submit(charger, f): f for f in fichiers}

    # Étape 2 : transformer dès que chargé
    transform_futures = {}
    for lf in as_completed(load_futures):
        data = lf.result()
        tf = cpu_pool.submit(transformer, data)
        transform_futures[tf] = data[0]

    # Étape 3 : sauvegarder dès que transformé
    save_futures = {}
    for tf in as_completed(transform_futures):
        result = tf.result()
        sf = io_pool.submit(sauvegarder, result)
        save_futures[sf] = result[0]

    for sf in as_completed(save_futures):
        print(f"  {sf.result()}")

print(f"Pipeline total : {time.perf_counter() - start:.2f}s")
```

</details>

---

## Ressources

- [docs Python — `concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html)
- [PEP 3148 — `futures`](https://peps.python.org/pep-3148/)
- [RealPython — ThreadPoolExecutor](https://realpython.com/python-concurrency/)
- *Fluent Python* (L. Ramalho), chapitre 20 — "Concurrent Executors"